# Step 1: Generate Synthetic Hotel Data

This notebook creates fake but realistic data for a multi-tenant Hotel Management System, so we have something to test our anomaly model on before touching real data.

We simulate **4 properties** over **26 weeks**, tracking:
- `properties` — the hotels themselves (our tenants)
- `occupancy` — how full each hotel is, week by week
- `products` — things hotels buy and use up (food, drinks, towels, cleaning chemicals)
- `suppliers` — who sells those products
- `purchases` — what was bought, from whom, at what price
- `stock_ledger` — how much stock came in, was used, and what's left

We will also **deliberately plant 4 anomalies** — one of each type we discussed (a stock discrepancy, a price spike, a consumption spike, and a rogue supplier). Later notebooks will try to catch them. We keep a separate "answer key" of what we planted, so we can check our work — but the detection logic itself will never be allowed to see this answer key.

In [ ]:
import numpy as np
import pandas as pd
import os

# Fixing the random seed means every time you run this notebook,
# you get the exact same 'random' data. Important for reproducibility.
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## Properties (our tenants)

Two bigger 'Tier1' hotels and two smaller 'Tier2' hotels, so we have real variety to prove that tenant-relative logic (comparing each hotel only to itself) actually matters.

In [ ]:
properties = pd.DataFrame([
    {'property_id': 'PR1', 'property_name': 'Grand Palace Hotel', 'city_tier': 'Tier1', 'rooms': 200},
    {'property_id': 'PR2', 'property_name': 'Seaside Resort',     'city_tier': 'Tier1', 'rooms': 150},
    {'property_id': 'PR3', 'property_name': 'City Business Inn',  'city_tier': 'Tier2', 'rooms': 80},
    {'property_id': 'PR4', 'property_name': 'Budget Stay Lodge',  'city_tier': 'Tier2', 'rooms': 50},
])
properties

## Occupancy

We generate 26 weeks of occupancy per property. Tier1 hotels run fuller on average than Tier2. We add a mild seasonal wave (sine curve) plus random noise, so occupancy looks realistic rather than a flat line.

This occupancy number is the 'business driver' we'll later compare consumption against — e.g. did food usage go up because the hotel was actually fuller, or is something odd going on?

In [ ]:
NUM_WEEKS = 26
start_date = pd.Timestamp('2026-02-02')  # a Monday
week_starts = [start_date + pd.Timedelta(weeks=i) for i in range(NUM_WEEKS)]

def gener ate_occupancy(properties, week_starts):
    rows = []
    for _, prop in properties.iterrows():
        base_occ = 0.72 if prop['city_tier'] == 'Tier1' else 0.55
        for week_num, week_start in enumerate(week_starts):
            seasonal = 0.08 * np.sin(2 * np.pi * week_num / 26)
            noise = np.random.normal(0, 0.04)
            occ = np.clip(base_occ + seasonal + noise, 0.2, 0.98)
            rows.append({
                'property_id': prop['property_id'],
                'week_num': week_num,
                'week_start': week_start,
                'occupancy_pct': round(occ, 3),
            })
    return pd.DataFrame(rows)

occupancy = generate_occupancy(properties, week_starts)
occupancy.head(10)

## Products and Suppliers

Six products across four categories (Food, Beverage, Housekeeping, Maintenance), and five suppliers. `SUP5` (QuickStock Traders) will later become our planted 'rogue supplier' — it doesn't normally supply anyone at the start.

In [ ]:
products = pd.DataFrame([
    {'product_id': 'PD1', 'product_name': 'Chicken Breast',    'category': 'Food',         'unit': 'kg'},
    {'product_id': 'PD2', 'product_name': 'Rice',               'category': 'Food',         'unit': 'kg'},
    {'product_id': 'PD3', 'product_name': 'Red Wine',           'category': 'Beverage',     'unit': 'bottle'},
    {'product_id': 'PD4', 'product_name': 'Beer',                'category': 'Beverage',    'unit': 'case'},
    {'product_id': 'PD5', 'product_name': 'Towels',              'category': 'Housekeeping','unit': 'unit'},
    {'product_id': 'PD6', 'product_name': 'Cleaning Chemicals',  'category': 'Maintenance', 'unit': 'litre'},
])

suppliers = pd.DataFrame([
    {'supplier_id': 'SUP1', 'supplier_name': 'Fresh Farms Co',       'category': 'Food'},
    {'supplier_id': 'SUP2', 'supplier_name': 'Global Beverages Ltd', 'category': 'Beverage'},
    {'supplier_id': 'SUP3', 'supplier_name': 'CleanPro Supplies',    'category': 'Housekeeping'},
    {'supplier_id': 'SUP4', 'supplier_name': 'Metro Wholesale',      'category': 'Maintenance'},
    {'supplier_id': 'SUP5', 'supplier_name': 'QuickStock Traders',   'category': 'Housekeeping'},
])

print(products)
print()
print(suppliers)

## Consumption and pricing rules, and the 4 planted anomalies

For each product we define:
- **base_rate_per_room_week** — how much of it a hotel normally uses per room per week
- **base_price** — its normal unit price
- **occupancy_driven** — whether usage should track occupancy (food/drinks/towels do; cleaning chemicals are roughly constant regardless of how full the hotel is)

Then we define the **4 anomalies we're deliberately injecting**, one of each type. We keep this list separate as an 'answer key' — later, our detection notebooks will try to find these on their own, without ever reading this list directly.

In [ ]:
product_config = {
    'PD1': {'base_rate_per_room_week': 0.9,  'base_price': 6.5,  'occupancy_driven': True},   # chicken, kg
    'PD2': {'base_rate_per_room_week': 0.6,  'base_price': 1.8,  'occupancy_driven': True},   # rice, kg
    'PD3': {'base_rate_per_room_week': 0.15, 'base_price': 12.0, 'occupancy_driven': True},   # wine, bottle
    'PD4': {'base_rate_per_room_week': 0.25, 'base_price': 28.0, 'occupancy_driven': True},   # beer, case
    'PD5': {'base_rate_per_room_week': 1.2,  'base_price': 4.0,  'occupancy_driven': True},   # towels, unit
    'PD6': {'base_rate_per_room_week': 0.3,  'base_price': 9.0,  'occupancy_driven': False},  # chemicals, litre
}

product_supplier_map = {
    'PD1': 'SUP1', 'PD2': 'SUP1',
    'PD3': 'SUP2', 'PD4': 'SUP2',
    'PD5': 'SUP3',
    'PD6': 'SUP4',
}

STARTING_STOCK = {
    'PD1': 50, 'PD2': 80, 'PD3': 40, 'PD4': 60, 'PD5': 300, 'PD6': 100,
}

PLANTED_ANOMALIES = [
    {'type': 'STOCK_DISCREPANCY', 'property_id': 'PR1', 'product_id': 'PD1', 'week_num': 15,
     'description': '25% of chicken breast stock goes missing at Grand Palace Hotel (shrinkage/theft)'},
    {'type': 'PRICE_ANOMALY', 'property_id': 'PR2', 'product_id': 'PD3', 'week_num': 10,
     'description': 'Red wine purchased at 3x normal price at Seaside Resort'},
    {'type': 'CONSUMPTION_ANOMALY', 'property_id': 'PR3', 'product_id': 'PD4', 'week_num': 20,
     'description': 'Beer consumption spikes 4x at City Business Inn despite normal occupancy'},
    {'type': 'SUPPLIER_ANOMALY', 'property_id': 'PR4', 'product_id': 'PD5', 'week_num': 18,
     'description': 'New supplier QuickStock Traders starts supplying towels to Budget Stay Lodge at 50% above normal price'},
]

pd.DataFrame(PLANTED_ANOMALIES)

## Running the simulation

For every property + product + week, we:
1. Compute how much was **consumed** this week (driven by occupancy, plus natural noise)
2. Compute how much was **purchased** to replenish it (roughly matches consumption, plus a small buffer)
3. Record the **purchase** (supplier + price)
4. Update the **stock ledger**: `opening + purchased - consumed = expected_closing`
5. Normally, the physically **counted_closing** stock matches `expected_closing` exactly. This is the accounting identity we'll check in the next notebook.

At the 4 specific points defined above, we override the normal numbers to inject our anomalies.

In [ ]:
def simulate(properties, occupancy, products, product_config, product_supplier_map):
    purchase_rows = []
    stock_rows = []

    occ_lookup = occupancy.set_index(['property_id', 'week_num'])['occupancy_pct'].to_dict()

    for _, prop in properties.iterrows():
        pid = prop['property_id']
        rooms = prop['rooms']

        for _, prod in products.iterrows():
            prod_id = prod['product_id']
            cfg = product_config[prod_id]
            default_supplier = product_supplier_map[prod_id]

            opening = STARTING_STOCK[prod_id]

            for week_num in range(NUM_WEEKS):
                occ = occ_lookup[(pid, week_num)]
                occ_factor = occ if cfg['occupancy_driven'] else 1.0

                # --- consumption ---
                consumed = cfg['base_rate_per_room_week'] * rooms * occ_factor
                consumed *= np.random.normal(1.0, 0.08)
                consumed = max(consumed, 0)

                # planted CONSUMPTION_ANOMALY
                if (pid, prod_id, week_num) == ('PR3', 'PD4', 20):
                    consumed *= 4.0

                # --- purchase (replenish what's used, plus a small buffer) ---
                purchased = consumed * np.random.normal(1.05, 0.05)
                purchased = max(purchased, 0)

                supplier_id = default_supplier
                unit_price = cfg['base_price'] * np.random.normal(1.0, 0.05)

                # planted SUPPLIER_ANOMALY: rogue supplier takes over from week 18 onward
                if (pid, prod_id) == ('PR4', 'PD5') and week_num >= 18:
                    supplier_id = 'SUP5'
                    unit_price = cfg['base_price'] * 1.5 * np.random.normal(1.0, 0.05)

                # planted PRICE_ANOMALY: one-off price spike
                if (pid, prod_id, week_num) == ('PR2', 'PD3', 10):
                    unit_price = cfg['base_price'] * 3.0

                purchase_rows.append({
                    'property_id': pid,
                    'product_id': prod_id,
                    'supplier_id': supplier_id,
                    'week_num': week_num,
                    'quantity': round(purchased, 2),
                    'unit_price': round(unit_price, 2),
                })

                # --- stock ledger ---
                expected_closing = opening + purchased - consumed
                counted_closing = expected_closing

                # planted STOCK_DISCREPANCY: 25% of stock goes missing, unrecorded
                if (pid, prod_id, week_num) == ('PR1', 'PD1', 15):
                    counted_closing = expected_closing * 0.75

                stock_rows.append({
                    'property_id': pid,
                    'product_id': prod_id,
                    'week_num': week_num,
                    'opening_qty': round(opening, 2),
                    'purchased_qty': round(purchased, 2),
                    'consumed_qty': round(consumed, 2),
                    'expected_closing_qty': round(expected_closing, 2),
                    'counted_closing_qty': round(counted_closing, 2),
                })

                # next week's opening stock = this week's actual counted closing stock
                opening = counted_closing

    return pd.DataFrame(purchase_rows), pd.DataFrame(stock_rows)

purchases, stock_ledger = simulate(properties, occupancy, products, product_config, product_supplier_map)
print('purchases:', purchases.shape)
print('stock_ledger:', stock_ledger.shape)
purchases.head(10)

In [ ]:
stock_ledger.head(10)

## Save everything to `/data`

We save each table as its own CSV. Later notebooks will read these back in — keeping data generation separate from detection logic is good practice, and lets you regenerate data without re-running the whole pipeline.

In [ ]:
os.makedirs('../data', exist_ok=True)

properties.to_csv('../data/properties.csv', index=False)
occupancy.to_csv('../data/occupancy.csv', index=False)
products.to_csv('../data/products.csv', index=False)
suppliers.to_csv('../data/suppliers.csv', index=False)
purchases.to_csv('../data/purchases.csv', index=False)
stock_ledger.to_csv('../data/stock_ledger.csv', index=False)
pd.DataFrame(PLANTED_ANOMALIES).to_csv('../data/planted_anomalies_answer_key.csv', index=False)

print('Saved all tables to /data\n')
for name, df in [('properties', properties), ('occupancy', occupancy), ('products', products),
                  ('suppliers', suppliers), ('purchases', purchases), ('stock_ledger', stock_ledger)]:
    print(f'{name}: {df.shape}')

## What's next

We now have a realistic multi-tenant dataset with 4 anomalies hidden inside it, and an answer key describing exactly what we planted (`planted_anomalies_answer_key.csv`) — for grading ourselves later, not for the model to see.

Next notebook: **Layer 1 — the stock reconciliation engine**, which checks `opening + purchased - consumed = closing` for every row and should catch the STOCK_DISCREPANCY we planted, with no ML involved at all.